# NeMo Customizer - Testing Customized Model

This notebook **tests** the customized model after it has been deployed to RHOAI's Single Serving Runtime (SSR). It does **not** download, merge, or upload the model—those steps use Python scripts from your machine (with port-forwards) because privileged `oc` or cluster access may not be available from the notebook.

## Prerequisites
- Completed training using `customize-model.ipynb`
- **Model deployment done via scripts** (from your machine): download adapter → merge with base → upload to MinIO. See README or `customize-model.ipynb` Step 18.
- InferenceService/SSR is configured to use the customized model path in MinIO
- `env.donotcommit` has `INFERENCE_SERVICE_URL`, `INFERENCE_SERVICE_NAME`, and optionally `CUSTOMIZED_MODEL_NAME`

## Step 1: Install Required Packages

In [ ]:
%pip install requests python-dotenv boto3

## Step 2: Import Libraries and Configure URLs

In [ ]:
import os
import json
import requests
from pathlib import Path

# Load environment variables from env.donotcommit if it exists
try:
    from dotenv import load_dotenv
    env_donotcommit_path = Path.cwd() / "env.donotcommit"
    if env_donotcommit_path.exists():
        load_dotenv(env_donotcommit_path, override=False)
        print(f"✅ Loaded env.donotcommit from: {env_donotcommit_path}")
    else:
        print(f"ℹ️  env.donotcommit not found at: {env_donotcommit_path}")
        print(f"   Using environment variables from system/defaults")
except ImportError:
    print("⚠️  python-dotenv not installed - using system environment variables only")

# Namespace configuration
NMS_NAMESPACE = os.getenv("NMS_NAMESPACE", "anemo-rhoai")

# Service URLs (cluster-internal)
CUSTOMIZER_URL = f"http://nemocustomizer-sample.{NMS_NAMESPACE}.svc.cluster.local:8000"
DATASTORE_URL = f"http://nemodatastore-sample.{NMS_NAMESPACE}.svc.cluster.local:8000"
ENTITY_STORE_URL = f"http://nemoentitystore-sample.{NMS_NAMESPACE}.svc.cluster.local:8000"

# API Keys and Tokens (from env.donotcommit)
NIM_SERVICE_ACCOUNT_TOKEN = os.getenv("NIM_SERVICE_ACCOUNT_TOKEN", "")

# Inference Service Configuration (for testing models)
INFERENCE_SERVICE_URL = os.getenv("INFERENCE_SERVICE_URL", "")
INFERENCE_SERVICE_NAME = os.getenv("INFERENCE_SERVICE_NAME", "")
CUSTOMIZED_MODEL_NAME = os.getenv("CUSTOMIZED_MODEL_NAME", "")

print(f"Namespace: {NMS_NAMESPACE}")
print(f"Inference Service URL: {INFERENCE_SERVICE_URL}")
print(f"Inference Service Name: {INFERENCE_SERVICE_NAME}")
print(f"Customized Model Name: {CUSTOMIZED_MODEL_NAME}")

## Model deployment (done by scripts, not this notebook)

Download, merge, and upload of the customized model are done **from your machine** using the Python scripts (they may use `oc` and port-forwards). The notebook cannot run privileged `oc` commands.

**From your machine:**
1. `python export_model_from_entity_store.py` (or `--job-id <id>`)
2. `python download_model_from_datastore.py --model-info model_info.json --output-dir ./downloaded_model`
3. `python merge_adapter_with_base.py --adapter-dir ./downloaded_model --output-dir ./merged_model`
4. Port-forward MinIO, then `MINIO_ENDPOINT=http://localhost:9000 python upload_model_to_minio.py --model-dir ./merged_model --target-path models/llama-3.2-1b-instruct-cust`
5. Update InferenceService to point to that MinIO path, then run the steps below to **test** the model.

## Step 18: Test Customized Model

Now let's test the customized model to verify that customization actually changed the model's behavior. We use the **exact same 3 prompts** as in `customize-model.ipynb` Step 12 (base model), and save responses in the **same file format** so you can compare side by side:

- **Base model** (from customize-model.ipynb Step 12): `base_model_responses.json`, `base_model_responses.txt`
- **Customized model** (this notebook): `customized_model_responses.json`, `customized_model_responses.txt`

In [ ]:
# Step 18.1: Check prerequisites and get customized model name

if not INFERENCE_SERVICE_URL:
    print("⚠️  INFERENCE_SERVICE_URL not set in env.donotcommit")
    print("   To test the customized model:")
    print("   1. Use Python scripts to download/merge/upload model to MinIO (see README)")
    print("   2. Update InferenceService to point to MinIO path, or create new InferenceService")
    print("   3. Set INFERENCE_SERVICE_URL to point to the InferenceService")
    print("   4. Set CUSTOMIZED_MODEL_NAME to the name of the customized model")
    print("   Example: INFERENCE_SERVICE_URL=http://your-customized-inference-service:8000")
    customized_model_name = None
else:
    # Get the customized model name
    customized_model_name = None
    if CUSTOMIZED_MODEL_NAME:
        customized_model_name = CUSTOMIZED_MODEL_NAME
    elif 'customized_model_name' in locals():
        customized_model_name = customized_model_name
    
    # Try to get model name from job details if available
    if not customized_model_name:
        print("⚠️  CUSTOMIZED_MODEL_NAME not set")
        print("   Please set CUSTOMIZED_MODEL_NAME in env.donotcommit")
        print("   Or enter it manually below")
        user_input = input("Enter customized model name (or press Enter to skip): ").strip()
        if user_input:
            customized_model_name = user_input
    
    if not customized_model_name:
        print("⚠️  Could not determine customized model name")
        print("   Please set CUSTOMIZED_MODEL_NAME in env.donotcommit")
        print("   Or ensure the customization job response contains 'output_model'")
    else:
        print(f"✅ Found customized model: {customized_model_name}")


In [ ]:
# Step 18.2: Detect working InferenceService URL for customized model

if not customized_model_name:
    print("⚠️  Skipping URL detection - customized model name not available")
    working_url = None
else:
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    
    headers = {
        "Content-Type": "application/json"
    }
    if NIM_SERVICE_ACCOUNT_TOKEN:
        headers["Authorization"] = f"Bearer {NIM_SERVICE_ACCOUNT_TOKEN}"
    
    potential_urls = [INFERENCE_SERVICE_URL]
    if INFERENCE_SERVICE_NAME:
        if INFERENCE_SERVICE_URL.startswith('https://'):
            internal_url = f"https://{INFERENCE_SERVICE_NAME}-predictor.{NMS_NAMESPACE}.svc.cluster.local:8443"
        else:
            internal_url = f"http://{INFERENCE_SERVICE_NAME}-predictor.{NMS_NAMESPACE}.svc.cluster.local:80"
        potential_urls.append(internal_url)
    
    model_names_to_try = []
    if INFERENCE_SERVICE_NAME:
        model_names_to_try.append(INFERENCE_SERVICE_NAME)
    model_names_to_try.append(customized_model_name)
    if "@" in customized_model_name:
        model_names_to_try.append(customized_model_name.split("@")[0])
    if "/" in customized_model_name:
        name_part = customized_model_name.split("/")[-1]
        model_names_to_try.append(name_part)
        if "@" in name_part:
            model_names_to_try.append(name_part.split("@")[0])
    model_names_to_try = list(dict.fromkeys(model_names_to_try))
    
    print(f"🔍 Detecting working InferenceService URL for customized model...")
    print(f"   Will try model names: {model_names_to_try}")
    
    working_url = None
    working_model_name = None
    
    for test_url in potential_urls:
        if working_url:
            break
        print(f"\n📡 Testing URL: {test_url}")
        for model_name in model_names_to_try:
            try:
                test_payload = {
                    "model": model_name,
                    "messages": [{"role": "user", "content": "test"}],
                    "max_tokens": 5
                }
                response = requests.post(
                    f"{test_url}/v1/chat/completions",
                    headers=headers,
                    json=test_payload,
                    timeout=10,
                    verify=False
                )
                if response.status_code == 200:
                    working_url = test_url
                    working_model_name = model_name
                    print(f"✅ Found working URL: {test_url}")
                    print(f"✅ Working model name: {model_name}")
                    break
                elif response.status_code == 503:
                    print(f"   ⚠️  Model '{model_name}' returned 503 (service unavailable)")
                elif response.status_code == 404:
                    print(f"   ⚠️  Model '{model_name}' not found (404)")
                else:
                    print(f"   ⚠️  Model '{model_name}' returned {response.status_code}: {response.text[:100]}")
            except requests.exceptions.SSLError as ssl_err:
                print(f"   ⚠️  SSL error with {test_url}: {str(ssl_err)[:80]}")
                break
            except Exception as e:
                print(f"   ⚠️  Error with model '{model_name}': {str(e)[:80]}")
                continue
    
    if not working_url:
        print("\n❌ No working URL found. Cannot test customized model.")
        print("\n💡 Troubleshooting steps:")
        print(f"   1. Check if InferenceService is running:")
        print(f"      oc get inferenceservice {INFERENCE_SERVICE_NAME} -n {NMS_NAMESPACE}")
        print(f"   2. Check if pods are ready:")
        print(f"      oc get pods -n {NMS_NAMESPACE} | grep {INFERENCE_SERVICE_NAME if INFERENCE_SERVICE_NAME else 'inference'}")
        print(f"   3. Verify the customized model is deployed:")
        print(f"      oc describe inferenceservice {INFERENCE_SERVICE_NAME} -n {NMS_NAMESPACE} | grep -i model")
        print(f"   4. Check service logs:")
        print(f"      oc logs -n {NMS_NAMESPACE} -l serving.kserve.io/inferenceservice={INFERENCE_SERVICE_NAME} --tail=50")
        print(f"   5. The customized model name is: {customized_model_name}")
        print(f"   6. Service URL: {INFERENCE_SERVICE_URL}")
        print(f"\n⚠️  Note: If you just created the InferenceService, it may take a few minutes")
        print(f"   for the customized model to load. Wait and try again.")
    else:
        print(f"\n✅ Ready to test! Using URL: {working_url}")
        print(f"   Model name: {working_model_name}")

In [ ]:
# Step 18.3: Test customized model with prompts

if not working_url:
    print("⚠️  Skipping customized model testing - no working URL found")
    print("   Run Step 18.2 first to detect a working InferenceService URL")
    customized_responses = {}
elif 'working_model_name' not in locals() or not working_model_name:
    print("⚠️  Skipping customized model testing - no working model name found")
    print("   Run Step 18.2 first to detect a working model name")
    customized_responses = {}
else:
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    
    headers = {
        "Content-Type": "application/json"
    }
    if NIM_SERVICE_ACCOUNT_TOKEN:
        headers["Authorization"] = f"Bearer {NIM_SERVICE_ACCOUNT_TOKEN}"
    
    test_prompts = [
        "What personal data does Red Hat collect?",
        "How does Red Hat use my personal data?",
        "What are my privacy rights with Red Hat?"
    ]
    print(f"📝 Using {len(test_prompts)} test prompts")
    
    customized_responses = {}
    
    print(f"\nTesting customized model responses using: {working_url}")
    print(f"Model name: {working_model_name}")
    print("=" * 60)
    
    for i, prompt in enumerate(test_prompts, 1):
        print(f"\n[{i}/{len(test_prompts)}] Prompt: {prompt[:50]}...")
        try:
            payload = {
                "model": working_model_name,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 200,
                "temperature": 0.7
            }
            
            response = requests.post(
                f"{working_url}/v1/chat/completions",
                headers=headers,
                json=payload,
                timeout=60,
                verify=False
            )
            
            if response.status_code == 200:
                result = response.json()
                if 'choices' in result and len(result['choices']) > 0:
                    response_text = result['choices'][0]['message']['content']
                    customized_responses[prompt] = response_text
                    print(f"✅ Response received ({len(response_text)} chars)")
                    print(f"   Preview: {response_text[:100]}...")
                else:
                    print(f"⚠️  Unexpected response format: {result}")
                    customized_responses[prompt] = None
            else:
                print(f"❌ Error: {response.status_code} - {response.text[:100]}")
                customized_responses[prompt] = None
        except Exception as e:
            print(f"❌ Exception: {str(e)[:100]}")
            customized_responses[prompt] = None
    
    print(f"\n" + "=" * 60)
    print(f"✅ Testing complete! Got {sum(1 for v in customized_responses.values() if v is not None)}/{len(test_prompts)} successful responses")
    # Save to same file format as base model (customize-model.ipynb) for easy comparison
    customized_responses_file = Path("customized_model_responses.json")
    customized_responses_txt = Path("customized_model_responses.txt")
    with open(customized_responses_file, "w", encoding="utf-8") as f:
        json.dump(customized_responses, f, indent=2, ensure_ascii=False)
    with open(customized_responses_txt, "w", encoding="utf-8") as f:
        for prompt, answer in customized_responses.items():
            f.write(f"Prompt: {prompt}\n")
            f.write(f"Response: {answer if answer is not None else '(no response)'}\n")
            f.write("-" * 60 + "\n")
    print(f"📁 Full responses saved to: {customized_responses_file.absolute()} and {customized_responses_txt.absolute()}")
    print(f"   Compare with base model: base_model_responses.json / base_model_responses.txt (from customize-model.ipynb Step 12)")

In [ ]:
# Step 18.4: Compare customized model responses with base model

if not customized_responses:
    print("⚠️  No customized model responses available for comparison")
    print("   Run Step 18.3 first to test the customized model")
else:
    print("📊 Response Comparison Summary")
    print("=" * 60)
    
    successful_responses = {k: v for k, v in customized_responses.items() if v is not None}
    
    if successful_responses:
        print(f"\n✅ Customized model returned {len(successful_responses)} successful response(s)")
        print(f"\n📝 Responses:")
        for prompt, response in successful_responses.items():
            print(f"\n{'='*60}")
            print(f"Prompt: {prompt}")
            print(f"{'-'*60}")
            print(f"Response:\n{response}")
            print(f"{'='*60}")
        
        print(f"\n💡 Analysis:")
        print(f"   - Review the responses above to verify customization worked")
        print(f"   - Compare with base model responses (if available) to see differences")
        print(f"   - Customized model should show changes in behavior based on training data")
    else:
        print(f"\n⚠️  No successful responses to display")
        print(f"   Check the errors above and verify:")
        print(f"   1. InferenceService is running and accessible")
        print(f"   2. Model name is correct: {working_model_name if 'working_model_name' in locals() else 'N/A'}")
        print(f"   3. Model is loaded and ready in the InferenceService")

## Summary - Testing Phase

This notebook completes the **testing** part of the customization workflow (deployment is done by Python scripts from your machine, not this notebook):

1. ✅ InferenceService URL detection (same prompts as base model)
2. ✅ Customized model testing with the 3 Red Hat privacy prompts
3. ✅ Responses saved to `customized_model_responses.json` and `.txt` for comparison with base
4. ✅ Side-by-side comparison of base vs customized model responses

**Model deployment** (download → merge → upload to MinIO) is done via scripts; this notebook only **tests** the model once SSR points to the customized path.

### Next Steps

1. Compare `base_model_responses.json` with `customized_model_responses.json` to validate customization
2. Adjust training data or parameters if needed and retrain
3. Deploy to production if results are satisfactory